# Job Task 2: Build the Vector Search source table

Writes `financial_docs_for_search` as a managed Delta table with Change Data Feed enabled.
This table is the source for the Vector Search Delta Sync index used by the Agent Bricks
Knowledge Assistant in Session 2.

**How it works:**
- `ai_prep_search()` takes the raw `ai_parse_document()` VARIANT (persisted in silver as
  `parsed_document`) and produces semantically chunked, context-enriched text optimised for
  vector search and embedding generation.
- Each document is exploded into multiple chunk rows.  `chunk_id` is the primary key;
  `chunk_to_embed` is the embedding source (enriched with titles, headers, page numbers);
  `chunk_text` is the raw chunk content returned to the user at query time.

Why a separate table rather than indexing `financial_docs_silver` directly?
- `financial_docs_silver` is a Streaming Table; Vector Search Delta Sync requires CDF,
  which is not compatible with SDP streaming tables.
- This job task reads from the completed silver table and writes a plain managed Delta table
  with `delta.enableChangeDataFeed = true`, making it a valid Delta Sync source.

In [0]:
dbutils.widgets.text("catalog",       "platform-workshop", "Catalog")
dbutils.widgets.text("shared_schema", "00_shared",         "Shared Schema")

catalog       = dbutils.widgets.get("catalog")
shared_schema = dbutils.widgets.get("shared_schema")

c = f"`{catalog}`"
print(f"Building search table in: {catalog}.{shared_schema}")

In [0]:
# Use ai_prep_search() to produce semantic chunks from the parsed document VARIANT.
# Each document explodes into multiple chunk rows — one per semantic segment.
#
# ai_prep_search() returns a VARIANT array of structs, each containing:
#   chunk_id        — unique identifier (document ID + chunk position)
#   chunk_to_embed  — context-enriched text (includes titles, headers, page numbers)
#   chunk_to_retrieve — raw chunk content for display at query time
#
# Metadata columns (company, fiscal_period, document_type) are carried through so
# they can be used as Vector Search filter dimensions.
from pyspark.sql.functions import col, explode, expr

chunks_df = (
    spark.read.table(f"{c}.{shared_schema}.financial_docs_silver")
    .filter("parsed_document IS NOT NULL")
    .withColumn(
        "chunks", 
        expr(
            "ai_prep_search(parsed_document):document:contents::ARRAY<STRUCT<chunk_id: STRING, chunk_to_embed: STRING, chunk_to_retrieve: STRING>>"
        )
    )
    .withColumn("chunk", explode("chunks"))
    .select(
        col("chunk.chunk_id").alias("chunk_id"),              # Primary key
        col("chunk.chunk_to_embed").alias("chunk_to_embed"),  # Embedding source
        col("chunk.chunk_to_retrieve").alias("chunk_text"),   # Display text
        "source_path",     # Document provenance
        "company",         # Metadata filter dimension
        "fiscal_period",   # Metadata filter dimension
        "document_type",   # Metadata filter dimension
    )
)

(
    chunks_df
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{c}.{shared_schema}.financial_docs_for_search")
)

print(f"✓ financial_docs_for_search written")

In [0]:
# Enable Change Data Feed — required for Vector Search Delta Sync.
# ALTER TABLE is idempotent; safe to run on every job execution.
spark.sql(f"""
    ALTER TABLE {c}.{shared_schema}.financial_docs_for_search
    SET TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'quality'                    = 'gold'
    )
""")

row_count = spark.sql(
    f"SELECT COUNT(*) AS cnt FROM {c}.{shared_schema}.financial_docs_for_search"
).collect()[0]['cnt']

doc_count = spark.sql(
    f"SELECT COUNT(DISTINCT source_path) AS cnt FROM {c}.{shared_schema}.financial_docs_for_search"
).collect()[0]['cnt']

print(f"✓ CDF enabled on financial_docs_for_search")
print(f"  Chunks available for indexing: {row_count:,} (from {doc_count:,} documents)")